CLICK ON THE LINK: [https://youtube.com/shorts/MP64xVguMco?si=Mvzr54ONxhYwu-L7]



## 1. Business Understanding

### 1.1 Business Problem

Driver drowsiness is a **major contributor to road accidents globally**, particularly in:

* Long-distance transportation
* Logistics and fleet operations
* Late-night and continuous shift driving

Fatigue leads to:

* Slower reaction time
* Reduced situational awareness
* Microsleep events

These factors directly cause:

* **Loss of life**
* **Vehicle damage**
* **Legal liabilities**
* **Operational downtime**
* **Insurance claims and rising premiums**

From a business perspective, drowsiness results in:

* Increased **accident cost per kilometer**
* Higher **fleet maintenance expenses**
* Reduced **driver productivity**
* Brand reputation risk

---

### 1.2 Business Objective

The primary objective of a Driver Drowsiness Detection System is to:

* **Detect early signs of driver fatigue in real time**
* **Alert the driver before a critical failure occurs**
* **Prevent accidents and save lives**
* **Reduce accident-related financial losses**
* **Improve fleet safety compliance**

Secondary objectives include:

* Enhancing driver performance monitoring
* Supporting insurance risk assessment
* Enabling regulatory safety compliance

---

### 1.3 Business Constraints

| Constraint Type              | Description                                                                                |
| ---------------------------- | ------------------------------------------------------------------------------------------ |
| **Cost Constraint**          | Hardware (camera, IR sensor), embedded systems, and cloud analytics increase cost.         |
| **Latency Constraint**       | Alerts must be generated in **real time (milliseconds)** to be effective.                  |
| **Environmental Constraint** | Performance must remain accurate in **low light, night driving, rain, and glare**.         |
| **Privacy Constraint**       | Captured facial data must comply with **data protection and consent regulations**.         |
| **Accuracy Constraint**      | High precision is mandatory to avoid **false alarms or missed detections**.                |
| **Hardware Constraint**      | System must run efficiently on **low-power edge devices** (Raspberry Pi, embedded boards). |




How IT Works: [https://youtu.be/8wx7g0BQmjs?si=aWVABrveTlvYnE2M]

## 2. What is Driver Drowsiness Detection (Technical Summary)

It is a **computer vision and AI-based safety system** that monitors:

* Eye closure (Eye Aspect Ratio – EAR)
* Blink duration
* Yawning frequency (Mouth Aspect Ratio – MAR)

Using:

* Camera input
* Facial landmark detection (MediaPipe / Dlib / OpenCV)
* Machine Learning or Deep Learning models

The system triggers:

* **Audio alerts**
* **Seat vibration**
* **Dashboard warnings**

## 3. Key Business Applications

### 3.1 Automotive Industry

* Integrated into **Advanced Driver Assistance Systems (ADAS)**
* Used by car manufacturers for **driver monitoring**

### 3.2 Fleet Management & Logistics

* Monitors truck drivers on:

  * Highways
  * Long-haul routes
* Reduces:

  * Insurance claims
  * Vehicle downtime
  * Fatal accidents

### 3.3 Public Transportation

* Used in:

  * Buses
  * Trains
  * Metro services
* Ensures **passenger safety**

### 3.4 Mining & Heavy Equipment

* Operators of:

  * Cranes
  * Excavators
  * Drilling machines
* Prevents fatigue-related industrial accidents

### 3.5 Ride-Hailing & Taxi Services

* Ensures driver alertness for:

  * Uber
  * Ola
  * Corporate taxi fleets

### 3.6 Defense & Aviation Ground Transport

* Used in:

  * Military transport vehicles
  * Airport ground vehicles




In [1]:
import sys

In [2]:
sys.version

'3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]'

In [3]:
#!pip install mediapipe

In [4]:
import mediapipe as mp
import cv2

In [5]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [10]:
model_path  = r"face_landmarker.task"    # Path of the .task file i.e.Reference Model

In [11]:
option = python.BaseOptions(model_asset_path=model_path)       # Load the Reference Model

In [12]:
final_option = vision.FaceLandmarkerOptions(base_options=option,
                            running_mode=vision.RunningMode.VIDEO,
                            num_faces = 1)                         # Configure

In [13]:
detector = vision.FaceLandmarker.create_from_options(final_option)    # Detecting the Face Landmarkers 

In [14]:
import math
import time

In [15]:
def right_eye_aspect_ratio(landmarks):
    p1 = landmarks[33].x ,landmarks[33].y
    p2 = landmarks[133].x ,landmarks[133].y

    a = math.dist(p1,p2)

    q1 = landmarks[160].x ,landmarks[160].y
    q2 = landmarks[144].x ,landmarks[144].y

    b = math.dist(q1,q2)

    r1 = landmarks[158].x ,landmarks[158].y
    r2 = landmarks[153].x ,landmarks[153].y 

    c = math.dist(r1,r2)

    rear = (b+c)/(2*a)

    return rear
    

In [16]:
def left_eye_aspect_ratio(landmarks):
    p1 = landmarks[362].x ,landmarks[362].y
    p2 = landmarks[263].x ,landmarks[263].y

    a = math.dist(p1,p2)

    q1 = landmarks[385].x ,landmarks[385].y
    q2 = landmarks[380].x ,landmarks[380].y

    b = math.dist(q1,q2)

    r1 = landmarks[387].x ,landmarks[387].y
    r2 = landmarks[373].x ,landmarks[373].y 

    c = math.dist(r1,r2)

    lear = (b + c) / (2*a)

    return(lear)

In [17]:
def Mouth_aspect_ratio(landmarks):
    p1 = landmarks[78].x ,landmarks[78].y
    p2 = landmarks[308].x ,landmarks[308].y

    a = math.dist(p1,p2)

    q1 = landmarks[81].x ,landmarks[81].y
    q2 = landmarks[178].x ,landmarks[178].y

    b = math.dist(q1,q2)

    r1 = landmarks[13].x ,landmarks[13].y
    r2 = landmarks[14].x ,landmarks[14].y 

    c = math.dist(r1,r2)

    s1 = landmarks[311].x ,landmarks[311].y
    s2 = landmarks[402].x ,landmarks[402].y 

    d = math.dist(s1,s2)

    mar = (b + c + d) / (3*a)

    return(mar)

In [18]:
time.time()

1791649263.1919284

In [19]:
import winsound

In [20]:
video = cv2.VideoCapture(0)

EAR_THRESHOLD = None
CALIBRATION_FRAMES = 30
calib_values = []

EAR_CLOSED_START = None
EAR_CLOSED_DURATION = 2.0   # seconds eyes must stay closed to trigger

MAR_THRESHOLD = 0.5
MAR_COUNTER = 0
MAR_FRAME = 8

while True:
    success, Frame = video.read()
    if not success:
        break

    image = cv2.cvtColor(Frame, cv2.COLOR_BGR2RGB)
    face = mp.Image(image_format=mp.ImageFormat.SRGB, data=image)
    time_frame = int(time.time() * 1000)
    result = detector.detect_for_video(image=face, timestamp_ms=time_frame)

    if result.face_landmarks:
        landmark = result.face_landmarks[0]
        LEAR = left_eye_aspect_ratio(landmark)
        REAR = right_eye_aspect_ratio(landmark)
        avg_ear = (LEAR + REAR) / 2
        mar = Mouth_aspect_ratio(landmark)

        if EAR_THRESHOLD is None:
            calib_values.append(avg_ear)
            cv2.putText(Frame, f'Calibrating... keep eyes open ({len(calib_values)}/{CALIBRATION_FRAMES})',
                        (20, 30), cv2.FONT_HERSHEY_TRIPLEX, 0.7, (0, 255, 0), 1)
            if len(calib_values) >= CALIBRATION_FRAMES:
                baseline = sum(calib_values) / len(calib_values)
                EAR_THRESHOLD = baseline * 0.75
                # print(f"Calibrated baseline EAR: {baseline:.3f}  -> threshold: {EAR_THRESHOLD:.3f}")
        else:
            cv2.putText(Frame, f'EAR : {round(avg_ear,2)}', (20,30),
                        cv2.FONT_HERSHEY_TRIPLEX, 1, (0,0,255), 1)
            cv2.putText(Frame, f'MAR : {round(mar,2)}', (20,60),
                        cv2.FONT_HERSHEY_TRIPLEX, 1, (255,0,0), 1)

            if avg_ear < EAR_THRESHOLD:
                if EAR_CLOSED_START is None:
                    EAR_CLOSED_START = time.time()
                eyes_closed_elapsed = time.time() - EAR_CLOSED_START
            else:
                EAR_CLOSED_START = None
                eyes_closed_elapsed = 0

            if mar > MAR_THRESHOLD:
                MAR_COUNTER += 1
            else:
                MAR_COUNTER = 0

            # print(f"EAR: {avg_ear:.3f} (thr {EAR_THRESHOLD:.3f})  eyes closed {eyes_closed_elapsed:.1f}s")

            if (eyes_closed_elapsed > EAR_CLOSED_DURATION) or (MAR_COUNTER > MAR_FRAME):
                cv2.putText(Frame, 'You are Drowzy', (50,100),
                            cv2.FONT_HERSHEY_TRIPLEX, 1, (0,0,255), 1)
                winsound.Beep(3000, 1000)

    cv2.imshow("Drowsiness Detection", Frame)
    if cv2.waitKey(100) == ord('a'):
        break

video.release()
cv2.destroyAllWindows()

## 4. Business Value & Impact

| Business Metric     | Impact                       |
| ------------------- | ---------------------------- |
| Accident Rate       | Reduced by 40–70%            |
| Insurance Cost      | Lower premiums               |
| Vehicle Maintenance | Lower repair cost            |
| Driver Safety       | Significantly improved       |
| Compliance          | Meets safety regulations     |
| Brand Trust         | Improved customer confidence |



---

## 5. Future Goals & Roadmap

### 5.1 AI & Deep Learning Enhancement

* Use of **CNN + LSTM models** for behavior prediction
* Personalized fatigue profiling per driver

### 5.2 Multi-Sensor Fusion

* Combine:

  * Eye tracking
  * Heart rate sensors
  * Steering behavior
  * Lane deviation detection

### 5.3 Integration with Smart Vehicles

* Fully connected with:

  * Autonomous driving systems
  * Smart dashboards
  * Cloud-based monitoring

### 5.4 Cloud & Fleet Analytics

* Centralized dashboards for:

  * Driver fatigue trends
  * High-risk route detection
  * Predictive accident analysis

### 5.5 Regulatory & Insurance Integration

* Direct integration with:

  * Road Transport Authorities
  * Insurance companies
* Enables:

  * Risk-based insurance premiums
  * Automated compliance reporting



---

## 6. Future Market Opportunity

* Global ADAS market is expanding rapidly.
* Growing demand from:

  * Electric Vehicles
  * Smart cities
  * AI-based safety enforcement
* High scope in:

  * India, Middle East, Southeast Asia


---

## 7. Summary (Executive View)

Driver Drowsiness Detection is a **high-impact, AI-driven safety system** that:

* Prevents fatal accidents
* Saves operational costs
* Improves fleet efficiency
* Ensures regulatory compliance
* Enhances road safety

It is a **critical foundation technology** for:

* Smart vehicles
* Autonomous driving
* Intelligent transportation systems

